# Demo - Prompting Techniques on a Local Model
**Day 2 - Session 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/demos/notebooks/demo-prompting-techniques.ipynb)

**Goal:** Show zero-shot, few-shot, and chain-of-thought prompting without any
API key, using a small instruction-tuned model that runs on the Colab CPU.

We use `google/flan-t5-base`, an instruction-following model, loaded directly
with transformers. The same techniques transfer directly to GPT, Claude, and
Gemini; only the client call changes. Decoding is greedy so the demo is
repeatable.

No API key needed. The model downloads once (~1 GB), then caches. For a
smaller download set `MODEL_NAME` to `"google/flan-t5-small"`.

In [ ]:
# Scoped install for this demo (safe to re-run; often preinstalled on Colab).
%pip install -q torch transformers sentencepiece

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "google/flan-t5-base"

In [ ]:
print(f"Loading {MODEL_NAME} (downloads once, then cached)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()


def make_generator(model, tokenizer):
    """Return a greedy text-in, text-out function backed by flan-t5."""
    def generate(prompt, max_new_tokens=64):
        inputs = tokenizer(prompt, return_tensors="pt")
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
        return tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return generate


def run(generate, label, prompt):
    """Print a labeled prompt and the model's answer."""
    answer = generate(prompt).strip()
    print(f"\n{label}")
    print(f"  prompt : {prompt}")
    print(f"  answer : {answer}")


generate = make_generator(model, tokenizer)

## 1. Zero-shot
No examples, rely on what the model already knows.

In [ ]:
def demo_zero_shot(generate):
    print("1. Zero-shot (no examples, rely on what the model knows)")
    run(
        generate,
        "Sentiment classification",
        "Classify the sentiment as POSITIVE, NEGATIVE, or NEUTRAL. "
        "Text: The update completely broke my workflow.",
    )


demo_zero_shot(generate)

## 2. Few-shot
A few examples teach the pattern and format.

In [ ]:
def demo_few_shot(generate):
    print("2. Few-shot (a few examples teach the pattern and format)")
    run(
        generate,
        "Sentiment with examples",
        "Classify the sentiment.\n"
        "Text: Best purchase ever. -> POSITIVE\n"
        "Text: Broke after one day. -> NEGATIVE\n"
        "Text: The battery could be better but I like it. ->",
    )


demo_few_shot(generate)

## 3. Chain-of-thought
Ask for step-by-step reasoning to fix a wrong direct answer on multi-step
math.

In [ ]:
def demo_chain_of_thought(generate):
    print("3. Chain-of-thought (ask for step-by-step reasoning)")
    run(
        generate,
        "Direct answer (small model slips on multi-step math)",
        "A week has 7 days. How many days are in 3 weeks?",
    )
    run(
        generate,
        "With 'think step by step'",
        "A week has 7 days. How many days are in 3 weeks? Think step by step.",
    )


demo_chain_of_thought(generate)

## Expected output
- Zero-shot returns a sentiment label, usually NEGATIVE, for the broken
  workflow text.
- Few-shot follows the `->` format shown in the examples and returns a label
  for the mixed-sentiment battery text.
- Chain-of-thought: the direct answer may be wrong or terse, while adding
  "Think step by step" often nudges the small model toward the correct answer
  (21 days) by walking through the arithmetic.

**Takeaway:** the technique lives in the prompt, not the provider. The same
zero/few-shot and CoT patterns work on any LLM API.